# 26 Spark SQL

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Use SQL and the DataFrame API interchangeably, write readable queries with CTEs and subqueries, pass parameters to SQL <b>safely</b>, and prove that both APIs compile to the same optimized plan.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
On Databricks, a large share of production pipelines, all of Databricks SQL, dashboards and many Lakeflow pipelines are written in SQL, and the certification exam shows most code as SQL. Strong data engineers move freely between SQL and PySpark, choosing whichever is clearer for each step. Building SQL with string formatting is also a classic security and bug risk, so parameter markers matter.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 One engine, two languages</b><br>
<code>spark.sql("...")</code> and DataFrame methods both produce a <b>logical plan</b> that the Catalyst optimizer turns into the same physical plan. There is no performance difference between them for the same logic.
<ul><li><code>spark.sql(query)</code> returns a <b>DataFrame</b>, so you can keep chaining Python methods</li><li>A DataFrame becomes queryable in SQL after <code>createOrReplaceTempView("name")</code> (lesson 27)</li><li><b>CTEs</b> (<code>WITH a AS (...)</code>) name intermediate steps. <b>Subqueries</b> nest queries (scalar, <code>IN</code>, <code>EXISTS</code>, correlated)</li><li><b>Parameter markers</b> (<code>:name</code>) pass values safely: <code>spark.sql("... WHERE x &gt; :min", args={"min": 10})</code></li><li><code>IDENTIFIER(:t)</code> turns a parameter into a table or column name safely</li></ul>
</div>

| Use SQL when... | Use the DataFrame API when... |
|---|---|
| The logic is set-based and reads naturally as a query | Logic is built dynamically (loops, configs, generated columns) |
| Analysts or SQL users will maintain it | You need reusable Python functions and unit tests |
| You use Databricks SQL, dashboards or `MERGE` | You mix in Python libraries or UDFs |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A retail team's Silver-to-Gold step is a 60-line SQL query with three CTEs (clean orders, latest customer record, daily totals), owned by analysts. The engineering team wraps it in a Python job that passes the processing date as a <b>parameter marker</b>, runs data quality checks on the resulting DataFrame in PySpark, and writes it to a Gold table. One plan, two languages.
</div>

## Syntax

```python
df = spark.sql("SELECT country, sum(amount) AS revenue FROM sales GROUP BY country")

spark.sql("SELECT * FROM sales WHERE sale_date = :d", args={"d": "2024-11-01"})   # named parameter
spark.sql("SELECT * FROM IDENTIFIER(:tbl)", args={"tbl": "workspace.spark_basics.sales"})
spark.sql("SELECT * FROM {df} WHERE amount > 100", df=some_dataframe)              # reference a DataFrame
```

```sql
WITH daily AS (
    SELECT sale_date, sum(amount) AS revenue FROM sales GROUP BY sale_date
)
SELECT * FROM daily WHERE revenue > (SELECT avg(revenue) FROM daily)
```

## Setup: sample data and views

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates customers and orders DataFrames and registers them as temporary views so SQL can see them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two small tables: 5 customers and 9 orders.
</div>

In [0]:
from pyspark.sql import functions as F

customers = spark.createDataFrame(
    [("C1", "Amal", "IN", "gold"), ("C2", "John", "UK", "silver"), ("C3", "Vivek", "AE", "gold"),
     ("C4", "Sara", "AE", "bronze"), ("C5", "Priya", "IN", "silver")],
    "customer_id STRING, name STRING, country STRING, tier STRING",
)
orders = spark.createDataFrame(
    [("O1", "C1", "2024-11-01", 120.0), ("O2", "C1", "2024-11-02", 80.0), ("O3", "C2", "2024-11-02", 300.0),
     ("O4", "C3", "2024-11-03", 45.0), ("O5", "C3", "2024-11-03", 500.0), ("O6", "C5", "2024-11-04", 60.0),
     ("O7", "C2", "2024-11-04", 20.0), ("O8", "C1", "2024-11-05", 210.0), ("O9", "C9", "2024-11-05", 99.0)],
    "order_id STRING, customer_id STRING, order_date STRING, amount DOUBLE",
).withColumn("order_date", F.to_date("order_date"))

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
customers.show(); orders.show()

## 1. The same query in SQL and in the DataFrame API

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes revenue per country for gold and silver customers in both languages, compares the results, and compares the optimized plans.<br><br>
<b>Why it matters</b><br>This settles the "is SQL slower than PySpark?" question with evidence: both produce the same physical plan.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Identical results (AE 545.0, IN 470.0, UK 320.0), then <code>True</code> for equal results and <code>True</code> for equal plans.
</div>

In [0]:
sql_df = spark.sql("""
    SELECT c.country, sum(o.amount) AS revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE c.tier IN ('gold', 'silver')
    GROUP BY c.country
    ORDER BY c.country
""")

api_df = (
    orders.alias("o")
    .join(customers.alias("c"), F.col("o.customer_id") == F.col("c.customer_id"))
    .filter(F.col("c.tier").isin("gold", "silver"))
    .groupBy("c.country")
    .agg(F.sum("o.amount").alias("revenue"))
    .orderBy("country")
)

sql_df.show()
print("Same results:", sql_df.collect() == api_df.collect())

def physical_plan(df):
    """The physical plan as text, with expression IDs (#123) removed so plans can be compared."""
    import re, io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return re.sub(r"#\d+L?|plan_id=\d+", "", buf.getvalue())

print("Same physical plan:", physical_plan(sql_df) == physical_plan(api_df))

## 2. Mixing SQL and Python

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Starts with SQL, continues with DataFrame methods, then queries a Python DataFrame from SQL without registering a view.<br><br>
<b>Why it matters</b><br><code>spark.sql</code> returns a DataFrame, and <code>spark.sql(..., df=...)</code> can reference a DataFrame directly with <code>{df}</code>. Use whichever language is clearer for each step.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Big orders (amount over 100) with a <code>vat</code> column added in Python, then the top customer by revenue computed in SQL from a Python DataFrame.
</div>

In [0]:
big_orders = spark.sql("SELECT * FROM orders WHERE amount > 100")          # SQL ...
big_orders.withColumn("vat", F.round(F.col("amount") * 0.2, 2)).show()   # ... then Python

revenue_by_customer = orders.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))
spark.sql(
    "SELECT customer_id, revenue FROM {rev} ORDER BY revenue DESC LIMIT 1",
    rev=revenue_by_customer,
).show()

## 3. CTEs: name each step

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a report in three named steps with a <code>WITH</code> clause: clean orders, customer totals, then a ranking.<br><br>
<b>Why it matters</b><br>CTEs make long queries readable top to bottom, like variables in Python. They don't materialize anything by themselves: the optimizer inlines them into one plan.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Customers with their total spend, order count and a rank. Orders for the unknown customer <code>C9</code> are excluded. Vivek ranks first with 545.0, then Amal with 410.0.
</div>

In [0]:
spark.sql("""
    WITH valid_orders AS (
        SELECT o.*
        FROM orders o
        WHERE EXISTS (SELECT 1 FROM customers c WHERE c.customer_id = o.customer_id)
    ),
    customer_totals AS (
        SELECT customer_id, sum(amount) AS total_spend, count(*) AS orders
        FROM valid_orders
        GROUP BY customer_id
    )
    SELECT c.name, t.total_spend, t.orders,
           dense_rank() OVER (ORDER BY t.total_spend DESC) AS spend_rank
    FROM customer_totals t
    JOIN customers c USING (customer_id)
    ORDER BY spend_rank
""").show()

## 4. Subqueries: scalar, IN, EXISTS and correlated

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the four common subquery forms.<br><br>
<b>Why it matters</b><br>Subqueries express "compared with the average", "customers who ordered" and "orders above the customer's own average" naturally. Spark rewrites most of them into joins (semi, anti, or aggregate joins).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
1) Orders above the overall average (about 159.3). 2) Customers with at least one order (C1, C2, C3, C5). 3) Customers without orders (C4). 4) Orders larger than that customer's own average.
</div>

In [0]:
# 1. Scalar subquery: one value
spark.sql("SELECT order_id, amount FROM orders WHERE amount > (SELECT avg(amount) FROM orders)").show()

# 2. IN subquery
spark.sql("SELECT customer_id, name FROM customers WHERE customer_id IN (SELECT customer_id FROM orders)").show()

# 3. NOT EXISTS (safer than NOT IN when the subquery can contain nulls)
spark.sql("""
    SELECT customer_id, name FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
""").show()

# 4. Correlated subquery: compares each row with its own group
spark.sql("""
    SELECT o.order_id, o.customer_id, o.amount
    FROM orders o
    WHERE o.amount > (SELECT avg(i.amount) FROM orders i WHERE i.customer_id = o.customer_id)
    ORDER BY o.customer_id
""").show()

## 5. Parameters: never build SQL with f-strings

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the dangerous way (string formatting with user input) and the safe way (named parameter markers and <code>IDENTIFIER</code>).<br><br>
<b>Why it matters</b><br>Formatting values into SQL strings breaks on quotes and allows <b>SQL injection</b>: a "country" value like <code>IN' OR '1'='1</code> returns every row. Parameter markers pass values as data, never as SQL text. <code>IDENTIFIER(:name)</code> does the same for table and column names. This is how Databricks jobs and dashboards pass parameters safely.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The f-string version with a malicious input returns all 5 customers. The parameterized version returns no rows for the same input, and 2 rows for <code>IN</code>. The <code>IDENTIFIER</code> query reads the table whose name was passed as a parameter.
</div>

In [0]:
user_input = "IN' OR '1'='1"

# Unsafe: the input becomes part of the SQL text
unsafe = spark.sql(f"SELECT name, country FROM customers WHERE country = '{user_input}'")
print("f-string rows:", unsafe.count())

# Safe: the input is passed as a value
safe = spark.sql("SELECT name, country FROM customers WHERE country = :country", args={"country": user_input})
print("parameter rows (malicious input):", safe.count())
spark.sql("SELECT name, country FROM customers WHERE country = :country", args={"country": "IN"}).show()

# Safe table names: IDENTIFIER() turns a parameter into an identifier
spark.sql("SELECT count(*) AS n FROM IDENTIFIER(:tbl)", args={"tbl": "orders"}).show()

## 6. Handy SQL-only conveniences

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows <code>GROUP BY ALL</code>, <code>count_if</code>, <code>FILTER</code> on aggregates, and <code>EXPLAIN</code> from SQL.<br><br>
<b>Why it matters</b><br>These make SQL shorter and are common in Databricks SQL. <code>GROUP BY ALL</code> groups by every non-aggregated column, so you never forget one.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Revenue and counts per country and tier, then the text of the physical plan from <code>EXPLAIN</code>.
</div>

In [0]:
spark.sql("""
    SELECT c.country, c.tier,
           sum(o.amount)                               AS revenue,
           count_if(o.amount > 100)                    AS big_orders,
           sum(o.amount) FILTER (WHERE o.amount < 100) AS small_order_revenue
    FROM orders o JOIN customers c USING (customer_id)
    GROUP BY ALL
    ORDER BY c.country, c.tier
""").show()

print(spark.sql("EXPLAIN SELECT country, count(*) FROM customers GROUP BY ALL").first()[0])

## Under the hood

```
spark.sql("SELECT ...")            df.filter(...).groupBy(...)...
        |                                       |
        v                                       v
   SQL parser -> unresolved plan      DataFrame API -> unresolved plan
                       \                   /
                        v                 v
               Analyzer (resolve tables, columns, types)
                              |
                              v
               Catalyst optimizer (same rules for both)
                              |
                              v
               Physical plan -> execution
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `spark.sql(...)` for a query is lazy: it returns a DataFrame without running it.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> DDL and DML statements (`CREATE`, `INSERT`, `DROP`, `MERGE`) run immediately when passed to `spark.sql`.

Subqueries are rewritten into joins: `IN`/`EXISTS` become **left semi** joins, `NOT EXISTS` becomes a **left anti** join, and correlated scalar subqueries become an aggregate plus a join. Look for `LeftSemi` and `LeftAnti` in the plans below.

In [0]:
spark.sql("SELECT * FROM customers c WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)").explain()
spark.sql("SELECT * FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>TABLE_OR_VIEW_NOT_FOUND</code> in <code>spark.sql</code></b><br>
The DataFrame wasn't registered as a view, the view was created in another session, or the catalog/schema is different. Register it, or use <code>spark.sql(..., df=df)</code> with <code>{df}</code>.<br><br>
<b>⛔ Problem: query breaks when a value contains a quote</b><br>You built SQL with an f-string. Use parameter markers (<code>:name</code> with <code>args=</code>).<br><br>
**⛔ Problem: <code>NOT IN</code> returns nothing**<br>The subquery returned a null. Use <code>NOT EXISTS</code>.<br><br>
**⛔ Problem: <code>[MISSING_AGGREGATION]</code> / "expression is neither present in the group by"**<br>Add the column to <code>GROUP BY</code>, aggregate it, or use <code>GROUP BY ALL</code>.<br><br>
<b>⛔ Problem: assuming SQL is slower than PySpark (or the reverse)</b><br>They compile to the same plan. Compare <code>explain()</code> outputs instead of guessing.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Is Spark SQL slower or faster than the DataFrame API?</b><br>
Neither. Both are parsed into a logical plan and optimized by Catalyst into the same physical plan, so the same logic performs the same. The choice is about readability and maintainability.<br><br>

<b>🎤 2. How do you run SQL on a DataFrame?</b><br>
Register it with <code>createOrReplaceTempView("name")</code> and query the view with <code>spark.sql</code>, or pass it directly with <code>spark.sql("SELECT ... FROM {df}", df=df)</code>.<br><br>

<b>🎤 3. What is a CTE and does it improve performance?</b><br>
A named subquery defined with <code>WITH</code>. It improves readability. Spark normally inlines CTEs into the plan, so it isn't materialized or faster by itself.<br><br>

<b>🎤 4. How does Spark execute an <code>EXISTS</code> or <code>IN</code> subquery?</b><br>
The optimizer rewrites it as a left semi join, and <code>NOT EXISTS</code> as a left anti join.<br><br>

<b>🎤 5. How do you pass parameters to Spark SQL safely?</b><br>
With named parameter markers, such as <code>:name</code> and <code>args={"name": value}</code>, and <code>IDENTIFIER(:tbl)</code> for table or column names. Never format user values into the SQL string, because that breaks on quotes and allows SQL injection.<br><br>

<b>🎤 6. Is <code>spark.sql</code> lazy?</b><br>
For queries, yes: it returns a DataFrame that runs on an action. DDL and DML statements such as <code>CREATE TABLE</code>, <code>INSERT</code> or <code>MERGE</code> execute immediately.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer has a DataFrame <code>df</code> in Python and wants to query it with SQL in the same notebook. What must they do first?</b><br>
A. Save it as a Delta table<br>
B. Register it with <code>df.createOrReplaceTempView("my_view")</code><br>
C. Convert it to pandas<br>
D. Nothing, SQL can see every Python variable<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Temp views make DataFrames visible to SQL in the same session. (<code>spark.sql</code> with a <code>{df}</code> placeholder is another option.)</details><br><br>

<b>Q2. Which query returns customers who have <b>no</b> orders, and handles NULL <code>customer_id</code> values in <code>orders</code> correctly?</b><br>
A. <code>SELECT * FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM orders)</code><br>
B. <code>SELECT * FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)</code><br>
C. <code>SELECT * FROM customers c JOIN orders o ON c.customer_id &lt;&gt; o.customer_id</code><br>
D. <code>SELECT * FROM customers EXCEPT ALL SELECT * FROM orders</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>NOT IN</code> returns no rows if the subquery contains a NULL. C produces a near-cross join, and D compares whole rows of different tables.</details><br><br>

<b>Q3. Which clause groups by every non-aggregated column in the SELECT list?</b><br>
A. <code>GROUP BY *</code><br>
B. <code>GROUP BY ALL</code><br>
C. <code>GROUP BY EVERY</code><br>
D. <code>GROUP BY AUTO</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using the <code>customers</code> and <code>orders</code> views:
<ol><li>With a CTE, compute each country's revenue and its share of total revenue (as a percentage), only for known customers</li><li>Write the same query with the DataFrame API and confirm the results match</li><li>Return the orders whose amount is above the average of their customer's <b>tier</b>, using a correlated subquery</li><li>Write a parameterized query that returns orders between two dates passed as parameters <code>:start</code> and <code>:end</code></li><li>In one sentence: why can't <code>IDENTIFIER(:tbl)</code> be replaced by <code>:tbl</code> alone?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. CTE with a share of total
q1 = spark.sql("""
    WITH by_country AS (
        SELECT c.country, sum(o.amount) AS revenue
        FROM orders o JOIN customers c USING (customer_id)
        GROUP BY c.country
    )
    SELECT country, revenue, round(revenue / (SELECT sum(revenue) FROM by_country) * 100, 1) AS pct
    FROM by_country
    ORDER BY country
""")
q1.show()

# 2. DataFrame version
by_country = orders.join(customers, "customer_id").groupBy("country").agg(F.sum("amount").alias("revenue"))
total = by_country.agg(F.sum("revenue")).first()[0]
q2 = by_country.withColumn("pct", F.round(F.col("revenue") / total * 100, 1)).orderBy("country")
print("Match:", q1.collect() == q2.collect())

# 3. Correlated subquery on tier
spark.sql("""
    SELECT o.order_id, c.tier, o.amount
    FROM orders o JOIN customers c USING (customer_id)
    WHERE o.amount > (
        SELECT avg(o2.amount) FROM orders o2 JOIN customers c2 USING (customer_id) WHERE c2.tier = c.tier
    )
    ORDER BY c.tier
""").show()

# 4. Parameters for a date range
spark.sql(
    "SELECT * FROM orders WHERE order_date BETWEEN :start AND :end ORDER BY order_date",
    args={"start": "2024-11-02", "end": "2024-11-03"},
).show()

# 5. A parameter marker is always a VALUE (a literal). Table and column names are identifiers,
#    so IDENTIFIER() is needed to safely turn the value into a name.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>SQL and the DataFrame API compile to the same optimized plan. Choose for readability</li><li><code>spark.sql</code> returns a DataFrame. Queries are lazy, DDL/DML runs immediately</li><li>Make DataFrames visible to SQL with temp views or <code>spark.sql(..., df=df)</code> and <code>{df}</code></li><li>CTEs name steps for readability. They are usually inlined, not materialized</li><li><code>IN</code>/<code>EXISTS</code> → left semi join, <code>NOT EXISTS</code> → left anti join. Avoid <code>NOT IN</code> with nullable subqueries</li><li>Use <code>:param</code> with <code>args=</code> and <code>IDENTIFIER(:name)</code>. Never f-strings with user input</li><li><code>GROUP BY ALL</code>, <code>count_if</code>, <code>FILTER (WHERE ...)</code> are handy SQL shortcuts</li></ul>
</div>

| Task | Code |
|---|---|
| Run SQL | `spark.sql("SELECT ...")` |
| SQL on a DataFrame | `spark.sql("SELECT * FROM {d}", d=df)` |
| Parameter | `spark.sql("... WHERE x = :v", args={"v": 1})` |
| Table name parameter | `SELECT * FROM IDENTIFIER(:tbl)` |
| CTE | `WITH a AS (...), b AS (...) SELECT ...` |
| Anti join in SQL | `WHERE NOT EXISTS (SELECT 1 FROM ... WHERE ...)` |
| Plan | `EXPLAIN SELECT ...` or `df.explain()` |

## Git commit

```
git add 01_spark_basics/26_spark_sql.ipynb
git commit -m "add 26 spark sql notebook"
```